<a href="https://colab.research.google.com/github/Marcin19721205/Timeseries_Data_Processing_Basic/blob/main/Section_7_ARn_models_with_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.graphics.tsaplots as sgt
import statsmodels.tsa.stattools as sts
from statsmodels.tsa.seasonal import seasonal_decompose
import seaborn as sns
sns.set()

In [4]:
# pip install numpy pandas plotly

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def simulate_ar(n: int, phi: np.ndarray, sigma2: float, seed: int = 123) -> np.ndarray:
    rng = np.random.default_rng(seed)
    p = len(phi)
    x = np.zeros(n, dtype=float)
    eps = rng.normal(0.0, np.sqrt(sigma2), size=n)

    # rozruch
    for k in range(p, n):
        x[k] = np.dot(phi, x[k-p:k][::-1]) + eps[k]
    return x

def generate_ts_smart(
    n=2000,
    noise_power=0.6,          # wariancja białego szumu dodawanego do sumy
    ar_phi=(0.65, -0.25, 0.18, 0.10),  # "prawdziwy" AR(4) do wykrycia przez PACF
    ar_noise_power=0.4,       # wariancja pobudzenia AR
    seasonal_period=48,       # okres sezonowości w próbkach
    seasonal_amp=3.0,         # skala sezonowości
    seed=123
):
    rng = np.random.default_rng(seed)
    t = np.arange(n)

    # 1) Trend: lekko nieliniowy + skok poziomu (żeby detrend było widać)
    trend = 0.002 * t + 2e-6 * (t - n/2)**2
    level_shift = np.where(t >= int(0.6*n), 3.0, 0.0)  # skok
    trend = trend + level_shift

    # 2) Sezonowość: powtarzalny "profil" (nie sinus) + modulacja amplitudy
    k = np.arange(seasonal_period)
    profile = (
        1.8 * np.exp(-0.5 * ((k - 0.25*seasonal_period)/(0.10*seasonal_period))**2)  # wąski pik
        - 0.9 * np.exp(-0.5 * ((k - 0.70*seasonal_period)/(0.18*seasonal_period))**2) # szeroka dolina
        + 0.2 * ((k/seasonal_period) - 0.5)  # lekka asymetria (rampa)
    )
    profile = (profile - profile.mean()) / (profile.std() + 1e-12)  # standaryzuj profil

    amp_mod = 1.0 + 0.25 * np.sin(2*np.pi * t / (10*seasonal_period))  # wolna modulacja amplitudy
    season = seasonal_amp * amp_mod * profile[t % seasonal_period]

    # 3) Składnik AR(p) - to potem ma wyjść w PACF i dać dobry AR(n)
    ar_part = simulate_ar(n, np.array(ar_phi, dtype=float), sigma2=ar_noise_power, seed=seed+1)

    # 4) Biały szum (sterujesz noise_power)
    noise = rng.normal(0.0, np.sqrt(noise_power), size=n)

    y = trend + season + ar_part + noise

    df = pd.DataFrame({
        "k": t,
        "y": y,
        "trend": trend,
        "season": season,
        "ar_part": ar_part,
        "noise": noise
    })
    return df

df = generate_ts_smart(
    n=2000,
    noise_power=0.6,
    ar_phi=(0.65, -0.25, 0.18, 0.10),
    ar_noise_power=0.4,
    seasonal_period=48,
    seasonal_amp=3.0,
    seed=123
)

# ---- wykres (dark) ----
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=["y (trend + season(profile) + AR + noise)", "Trend + Season", "AR part (ukryty składnik)"]
)

fig.add_trace(go.Scatter(x=df["k"], y=df["y"], mode="lines", name="y"), row=1, col=1)

fig.add_trace(go.Scatter(x=df["k"], y=df["trend"], mode="lines", name="trend"), row=2, col=1)
fig.add_trace(go.Scatter(x=df["k"], y=df["season"], mode="lines", name="season"), row=2, col=1)

fig.add_trace(go.Scatter(x=df["k"], y=df["ar_part"], mode="lines", name="ar_part"), row=3, col=1)

fig.update_layout(template="plotly_dark", height=900, width=1100, legend=dict(orientation="h"))
fig.update_xaxes(title_text="k", row=3, col=1)
fig.update_yaxes(title_text="value", row=1, col=1)
fig.show()


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   k        2000 non-null   int64  
 1   y        2000 non-null   float64
 2   trend    2000 non-null   float64
 3   season   2000 non-null   float64
 4   ar_part  2000 non-null   float64
 5   noise    2000 non-null   float64
dtypes: float64(5), int64(1)
memory usage: 93.9 KB


In [6]:
import pandas as pd

# zakładam, że masz już df z kolumną "y" (np. z generate_ts_smart)

df_y = df.copy()

# Time as Index (minutowe próbkowanie)
df_y["Time"] = pd.date_range(
    start="2026-01-01 00:00:00",  # dowolny start
    periods=len(df_y),
    freq="min"
)

df_y["Time"] = pd.to_datetime(df_y["Time"])   # tu akurat już jest datetime, ale trzymamy styl jak w wzorcu
df_y.set_index("Time", inplace=True)

df_y = df_y[["y"]]            # wytnij niepotrzebne kolumny (zostaje tylko wynik)
df_y = df_y.asfreq("min")     # data frequency as 'minute'
df_y = df_y.ffill()           # fill NaN forward (gdyby asfreq coś wstawił)

df_y.head()


,y
Time,
2026-01-01 00:00:00,0.930093
2026-01-01 00:01:00,1.607782
2026-01-01 00:02:00,3.174741
2026-01-01 00:03:00,2.721027
2026-01-01 00:04:00,4.314876


In [7]:
df_y.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2000 entries, 2026-01-01 00:00:00 to 2026-01-02 09:19:00
Freq: min
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   y       2000 non-null   float64
dtypes: float64(1)
memory usage: 31.2 KB


In [8]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_y.index, y=df_y["y"], mode="lines", name="y"))

fig.update_layout(
    template="plotly_dark",
    title="y(t) — 1-min sampling",
    xaxis_title="Time",
    yaxis_title="y"
)

fig.show()


ACF dla sezonowości

In [9]:
# pip install numpy statsmodels plotly

import numpy as np
from statsmodels.tsa.stattools import acf
import plotly.graph_objects as go

# --- wejście ---
y = df_y["y"].astype(float).values

# --- parametry ACF ---
max_lag = 600      # ile lagów liczysz (minuty)
min_period = 10    # min sensowny okres
max_period = 300   # max sensowny okres

acf_vals = acf(y, nlags=max_lag, fft=True)
lags = np.arange(max_lag + 1)

# --- estymacja okresu z ACF (RAW) ---
lo, hi = min_period, min(max_period, max_lag)
search = acf_vals[lo:hi+1]
period_est = int(lo + np.argmax(search))

top_k = 5
top_idx = np.argsort(search)[::-1][:top_k]
top_periods = (lo + top_idx).tolist()
top_vals = search[top_idx].tolist()

print(f"Estimated period from RAW ACF: {period_est} samples (minutes)")
print("Top candidates:", list(zip(top_periods, [round(v,4) for v in top_vals])))

# --- wykres ACF (Plotly dark) ---
fig = go.Figure()
fig.add_trace(go.Bar(x=lags, y=acf_vals, name="ACF(y)"))
fig.add_hline(y=0)
fig.add_vline(x=period_est)

fig.update_layout(
    template="plotly_dark",
    title=f"ACF(y) — RAW | period_est={period_est}",
    xaxis_title="lag [min]",
    yaxis_title="ACF",
    height=500,
    width=1100,
    showlegend=False
)
fig.show()


Estimated period from RAW ACF: 48 samples (minutes)
Top candidates: [(48, 0.8845), (47, 0.8789), (49, 0.876), (46, 0.8598), (50, 0.856)]


Detrening - usunięcie sezonowości

estymowny period wpisz w zmienną: seasonal_period

In [10]:
# pip install statsmodels plotly

import pandas as pd
from statsmodels.tsa.seasonal import STL
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# df_y: DataFrame z DateTimeIndex (co minutę) i jedną kolumną: "y"

seasonal_period = 48  # <- u

y = df_y["y"].astype(float)

stl = STL(y, period=seasonal_period, robust=True)
stl_res = stl.fit()

# składowe
y_trend   = stl_res.trend.rename("trend")
y_season  = stl_res.seasonal.rename("seasonal")
y_resid   = stl_res.resid.rename("resid")                 # to jest "po detrend i po sezonowości"
y_detr    = (y - y_trend).rename("y_detrended")           # usunięty trend, sezon zostaje
y_deseas  = (y - y_season).rename("y_deseasonalized")     # usunięta sezonowość, trend zostaje

# (opcjonalnie) wrzuć do jednego DF, bo potem przy ACF/PACF wygodniej
df_clean = pd.concat([y, y_trend, y_season, y_detr, y_deseas, y_resid], axis=1)

# ----------------------------
# Wykresy (Plotly dark)
# ----------------------------
fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=[
        "Oryginał y oraz trend (STL)",
        "Sezonowość (STL)",
        "y po detrend (y - trend) i po deseason (y - seasonal)",
        "Residual (y - trend - seasonal)  — dla modeli ACF/PACF/AR"
    ]
)

# 1) y + trend
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["y"], mode="lines", name="y"), row=1, col=1)
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["trend"], mode="lines", name="trend"), row=1, col=1)

# 2) seasonal
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["seasonal"], mode="lines", name="seasonal"), row=2, col=1)

# 3) detrended + deseasonalized
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["y_detrended"], mode="lines", name="y_detrended"), row=3, col=1)
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["y_deseasonalized"], mode="lines", name="y_deseasonalized"), row=3, col=1)

# 4) residual
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["resid"], mode="lines", name="resid"), row=4, col=1)

fig.update_layout(template="plotly_dark", height=1050, width=1200, legend=dict(orientation="h"))
fig.update_xaxes(title_text="Time", row=4, col=1)
fig.show()

# df_clean.head()


In [11]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2000 entries, 2026-01-01 00:00:00 to 2026-01-02 09:19:00
Freq: min
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   y                 2000 non-null   float64
 1   trend             2000 non-null   float64
 2   seasonal          2000 non-null   float64
 3   y_detrended       2000 non-null   float64
 4   y_deseasonalized  2000 non-null   float64
 5   resid             2000 non-null   float64
dtypes: float64(6)
memory usage: 109.4 KB


ADF - test

In [12]:
import statsmodels.tsa.stattools as sts  # ADF test lives here

# --- columns to test (choose what makes sense) ---
cols = ["y", "y_detrended", "y_deseasonalized", "resid"]
alpha = 0.05

# sanity: keep only columns that exist
cols = [c for c in cols if c in df_clean.columns]

print(f"Running ADF for {len(cols)} columns: {cols}")

for col in cols:
    print("\n" + "=" * 80)
    print(f"ADF test | column: {col}")

    series = df_clean[col].dropna()

    adf_out = sts.adfuller(series, autolag="AIC")

    adf_stat = adf_out[0]
    p_value  = adf_out[1]
    used_lags = adf_out[2]
    n_obs = adf_out[3]
    crit = adf_out[4]
    icbest = adf_out[5]

    print(f"ADF statistic: {adf_stat:.6f}")
    print(f"p-value:       {p_value:.6g}")
    print(f"used lags:     {used_lags}")
    print(f"n obs:         {n_obs}")
    print(f"critical values: 1%={crit['1%']:.6f}, 5%={crit['5%']:.6f}, 10%={crit['10%']:.6f}")
    print(f"IC best:       {icbest:.6f}")

    if p_value < alpha:
        print(f"Decision @ {alpha}: REJECT H0 (unit root) -> series is likely STATIONARY.")
    else:
        print(f"Decision @ {alpha}: FAIL TO REJECT H0 -> series is likely NON-STATIONARY (unit root).")

    if adf_stat < crit["5%"]:
        print("Sanity check: ADF statistic is below the 5% critical value -> supports stationarity.")
    else:
        print("Sanity check: ADF statistic is NOT below the 5% critical value -> supports non-stationarity.")


Running ADF for 4 columns: ['y', 'y_detrended', 'y_deseasonalized', 'resid']

ADF test | column: y
ADF statistic: -4.986682
p-value:       2.35803e-05
used lags:     26
n obs:         1973
critical values: 1%=-3.433669, 5%=-2.863006, 10%=-2.567550
IC best:       6657.762362
Decision @ 0.05: REJECT H0 (unit root) -> series is likely STATIONARY.
Sanity check: ADF statistic is below the 5% critical value -> supports stationarity.

ADF test | column: y_detrended
ADF statistic: -18.688363
p-value:       2.04e-30
used lags:     26
n obs:         1973
critical values: 1%=-3.433669, 5%=-2.863006, 10%=-2.567550
IC best:       6356.158267
Decision @ 0.05: REJECT H0 (unit root) -> series is likely STATIONARY.
Sanity check: ADF statistic is below the 5% critical value -> supports stationarity.

ADF test | column: y_deseasonalized
ADF statistic: -0.569620
p-value:       0.8777
used lags:     19
n obs:         1980
critical values: 1%=-3.433657, 5%=-2.863001, 10%=-2.567548
IC best:       5330.166427

#ADF
Biorę już tylko resid dla dalszych

In [13]:
import statsmodels.tsa.stattools as sts

alpha = 0.05
series = df_clean["resid"].dropna()

adf_stat, p_value, used_lags, n_obs, crit, icbest = sts.adfuller(series, autolag="AIC")

print("ADF test | series: resid")
print(f"ADF statistic: {adf_stat:.6f}")
print(f"p-value:       {p_value:.6g}")
print(f"used lags:     {used_lags}")
print(f"n obs:         {n_obs}")
print(f"critical values: 1%={crit['1%']:.6f}, 5%={crit['5%']:.6f}, 10%={crit['10%']:.6f}")
print(f"IC best:       {icbest:.6f}")

if p_value < alpha:
    print(f"Decision @ {alpha}: REJECT H0 (unit root) -> resid is likely STATIONARY.")
else:
    print(f"Decision @ {alpha}: FAIL TO REJECT H0 -> resid is likely NON-STATIONARY.")


ADF test | series: resid
ADF statistic: -10.747159
p-value:       2.72322e-19
used lags:     22
n obs:         1977
critical values: 1%=-3.433662, 5%=-2.863003, 10%=-2.567549
IC best:       5195.714024
Decision @ 0.05: REJECT H0 (unit root) -> resid is likely STATIONARY.


PACF

In [14]:
# ===== PACF(resid) + 95% CI band + wybór rzędu AR(p) =====
# pip install numpy pandas statsmodels plotly scipy

import numpy as np
import pandas as pd

from scipy.stats import norm
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import pacf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------------------------------------------------------
# 0) Wejście: df_y (DateTimeIndex co minutę, kolumna "y")
# ---------------------------------------------------------
# df_y musi istnieć wcześniej. Jeśli nie, to:
# df_y = ...  # DataFrame z kolumną "y" i indexem datetime freq="min"

# ---------------------------------------------------------
# 1) STL -> df_clean z resid
# ---------------------------------------------------------
seasonal_period = 48  # <-- ustaw (np. z ACF). 48 = 48 minut

y = df_y["y"].astype(float)
stl_res = STL(y, period=seasonal_period, robust=True).fit()

df_clean = pd.concat([
    y.rename("y"),
    stl_res.trend.rename("trend"),
    stl_res.seasonal.rename("seasonal"),
    (y - stl_res.trend).rename("y_detrended"),
    (y - stl_res.seasonal).rename("y_deseasonalized"),
    stl_res.resid.rename("resid")
], axis=1)

# ---------------------------------------------------------
# 2) PACF(resid) + 95% CI (±z/sqrt(N)) + wybór p
# ---------------------------------------------------------
x = df_clean["resid"].dropna().astype(float).values
n = len(x)

max_lag = 40
alpha = 0.05  # 95% CI

pacf_vals = pacf(x, nlags=max_lag, method="ywmle")

z = norm.ppf(1 - alpha/2)
conf = z / np.sqrt(n)  # 95% band around 0

lags = np.arange(max_lag + 1)

sig_lags = np.where(np.abs(pacf_vals[1:]) > conf)[0] + 1  # ignore lag 0
p_ar = int(sig_lags.max()) if len(sig_lags) else 0

print(f"PACF significance band: ±{conf:.6f} (N={n})")
print(f"Significant lags: {sig_lags.tolist()}")
print(f"Chosen AR order p = {p_ar}")

# ---------------------------------------------------------
# 3) Plotly dark: resid + PACF z CI
# ---------------------------------------------------------
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=False, vertical_spacing=0.10,
    subplot_titles=[
        "resid (po STL: y - trend - seasonal)",
        f"PACF(resid) | 95% CI band ±{conf:.4f} | chosen p={p_ar}"
    ]
)

# resid
fig.add_trace(go.Scatter(
    x=df_clean.index, y=df_clean["resid"],
    mode="lines", name="resid"
), row=1, col=1)

# PACF bars
fig.add_trace(go.Bar(
    x=lags, y=pacf_vals, name="PACF"
), row=2, col=1)

# CI lines (bardziej czytelne niż prostokąt)
fig.add_hline(y=0, row=2, col=1)
fig.add_hline(y= conf, row=2, col=1)
fig.add_hline(y=-conf, row=2, col=1)

# chosen p
if p_ar > 0:
    fig.add_vline(x=p_ar, row=2, col=1)

fig.update_layout(
    template="plotly_dark",
    height=850,
    width=1200,
    legend=dict(orientation="h")
)

fig.update_xaxes(title_text="Time", row=1, col=1)
fig.update_xaxes(title_text="lag", row=2, col=1)
fig.update_yaxes(title_text="value", row=1, col=1)
fig.update_yaxes(title_text="PACF", row=2, col=1)

fig.show()


PACF significance band: ±0.043826 (N=2000)
Significant lags: [1, 2, 4, 16, 18, 21]
Chosen AR order p = 21


PACF - sensowny

In [15]:
# ===== PACF(resid) + 95% CI band + wybór rzędu AR(p) =====
# pip install numpy pandas statsmodels plotly scipy

import numpy as np
import pandas as pd

from scipy.stats import norm
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import pacf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------------------------------------------------------
# 0) Wejście: df_y (DateTimeIndex co minutę, kolumna "y")
# ---------------------------------------------------------
# df_y musi istnieć wcześniej. Jeśli nie, to:
# df_y = ...  # DataFrame z kolumną "y" i indexem datetime freq="min"

# ---------------------------------------------------------
# 1) STL -> df_clean z resid
# ---------------------------------------------------------
seasonal_period = 48  # <-- ustaw (np. z ACF). 48 = 48 minut

y = df_y["y"].astype(float)
stl_res = STL(y, period=seasonal_period, robust=True).fit()

df_clean = pd.concat([
    y.rename("y"),
    stl_res.trend.rename("trend"),
    stl_res.seasonal.rename("seasonal"),
    (y - stl_res.trend).rename("y_detrended"),
    (y - stl_res.seasonal).rename("y_deseasonalized"),
    stl_res.resid.rename("resid")
], axis=1)

# ---------------------------------------------------------
# 2) PACF(resid) + 95% CI (±z/sqrt(N)) + wybór p
# ---------------------------------------------------------
x = df_clean["resid"].dropna().astype(float).values
n = len(x)

max_lag = 10
alpha = 0.05  # 95% CI

pacf_vals = pacf(x, nlags=max_lag, method="ywmle")

z = norm.ppf(1 - alpha/2)
conf = z / np.sqrt(n)  # 95% band around 0

lags = np.arange(max_lag + 1)

sig_lags = np.where(np.abs(pacf_vals[1:]) > conf)[0] + 1  # ignore lag 0
p_ar = int(sig_lags.max()) if len(sig_lags) else 0

print(f"PACF significance band: ±{conf:.6f} (N={n})")
print(f"Significant lags: {sig_lags.tolist()}")
print(f"Chosen AR order p = {p_ar}")

# ---------------------------------------------------------
# 3) Plotly dark: resid + PACF z CI
# ---------------------------------------------------------
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=False, vertical_spacing=0.10,
    subplot_titles=[
        "resid (po STL: y - trend - seasonal)",
        f"PACF(resid) | 95% CI band ±{conf:.4f} | chosen p={p_ar}"
    ]
)

# resid
fig.add_trace(go.Scatter(
    x=df_clean.index, y=df_clean["resid"],
    mode="lines", name="resid"
), row=1, col=1)

# PACF bars
fig.add_trace(go.Bar(
    x=lags, y=pacf_vals, name="PACF"
), row=2, col=1)

# CI lines (bardziej czytelne niż prostokąt)
fig.add_hline(y=0, row=2, col=1)
fig.add_hline(y= conf, row=2, col=1)
fig.add_hline(y=-conf, row=2, col=1)

# chosen p
if p_ar > 0:
    fig.add_vline(x=p_ar, row=2, col=1)

fig.update_layout(
    template="plotly_dark",
    height=850,
    width=1200,
    legend=dict(orientation="h")
)

fig.update_xaxes(title_text="Time", row=1, col=1)
fig.update_xaxes(title_text="lag", row=2, col=1)
fig.update_yaxes(title_text="value", row=1, col=1)
fig.update_yaxes(title_text="PACF", row=2, col=1)

fig.show()


PACF significance band: ±0.043826 (N=2000)
Significant lags: [1, 2, 4]
Chosen AR order p = 4
